In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "HuggingFaceTB/SmolLM3-3B"
device = "cuda"  # for GPU usage or "cpu" for CPU usage

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.to(device)

# prepare the model input
prompt = "Give me a brief explanation of gravity in simple terms in Italian language."
messages_think = [
    #{"role": "user", "content": "You are an helpful assistant"}
    {"role": "user", "content": prompt}
]
#La lista di messaggi è una lista di dizionari {"role": user o system o assystant, "content": Il contenuto}



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

1. Parametri Standard (Della libreria transformers)
Questi funzionano con quasi tutti i tokenizer moderni.

conversation (o primo argomento posizionale):

Cosa è: La lista di dizionari con i messaggi (es. [{"role": "user", "content": "..."}]).

Obbligatorio: Sì.

tokenize (bool, default True):

Cosa fa: Se True, restituisce i numeri (input_ids). Se False, restituisce una stringa (il testo formattato con i tag speciali).

Uso: Mettilo a False se vuoi vedere come viene costruito il prompt (debug) o se devi fare operazioni sulla stringa prima di tokenizzare.

add_generation_prompt (bool, default False):

Cosa fa: Se True, aggiunge alla fine il token che indica l'inizio della risposta dell'assistente (es. <|assistant|> o <start_of_turn>model).

Uso: Fondamentale quando stai facendo inferenza/generazione. Se stai facendo training (fine-tuning), di solito lo metti a False.

continue_final_message (bool, default False):

Cosa fa: Se l'ultimo messaggio è dell'assistente, permette di continuare quella risposta invece di iniziarne una nuova.

Uso: Utile per il pre-filling (es. forzare l'AI a iniziare la risposta con "Ecco il codice JSON:").

chat_template (str, opzionale):

Cosa fa: Ti permette di passare una stringa Jinja2 personalizzata, ignorando quella salvata nel tokenizer (tokenizer.chat_template).

Uso: Se vuoi cambiare completamente il formato dei prompt al volo.

return_tensors (str, opzionale):

Cosa fa: Se tokenize=True, specifica il tipo di tensore: 'pt' (PyTorch), 'tf' (TensorFlow), 'np' (NumPy).

tools (list, opzionale):

Cosa fa: Passa la definizione delle funzioni/tool disponibili per il Function Calling.

Uso: Il template deve supportare i tool (modelli come Llama 3 o Mistral lo fanno).

documents (list, opzionale):

Cosa fa: Per la RAG (Retrieval Augmented Generation), passa i documenti di contesto che il template può formattare automaticamente.

2. Parametri Dinamici (**kwargs) - La "Magia"
Qui è dove rientra il tuo enable_thinking=False. Il metodo accetta **kwargs, il che significa che qualsiasi parametro extra che passi viene inviato direttamente al motore Jinja2.

Se il chat_template del modello contiene una logica come questa:

Django
{% if enable_thinking %}
   <think> ... </think>
{% endif %}
Allora passando enable_thinking=True attivi quella parte.

Esempi di parametri dinamici comuni (dipendono dal modello!):

system_message: Alcuni template permettono di sovrascrivere il system prompt passando questo parametro, anche se non è nella lista dei messaggi.

use_default_system_prompt: (Comune in Llama 2/3) Booleano per decidere se includere il prompt di sistema di default.

date / datetime: Alcuni template richiedono la data corrente per dare contesto temporale al modello.

Esempio completo e avanzato
Ecco come potresti usare apply_chat_template sfruttando sia i parametri standard che quelli dinamici (ipotizzando un modello DeepSeek-R1 o simile):

Python
text = tokenizer.apply_chat_template(
    conversation=messages,
    
    # --- Parametri Standard Transformers ---
    tokenize=False,              # Voglio la stringa, non i numeri
    add_generation_prompt=True,  # Prepara il modello a rispondere
    continue_final_message=False,# Non sto continuando una frase a metà
    tools=tools_list,            # Passo la lista dei tool (se il modello li supporta)
    
    # --- Parametri Dinamici (passati a Jinja) ---
    enable_thinking=True,        # Attivo i tag <think> (specifico del tuo modello)
    date="2023-10-27"            # Se il template supporta una data odierna custom
)

 Output ipotetico (stringa):
 <|begin_of_text|><|start_header_id|>user<|end_header_id|>
 Ciao!<|eot_id|><|start_header_id|>assistant<|end_header_id|>

 <think>

In [3]:
text = tokenizer.apply_chat_template(
    #Parametri standard conversation, tokenize, add_generation_prompt
    messages_think,
    tokenize=False,
    add_generation_prompt=True,
    # Parametri dinamici passati a jinja. Per scoprire questi parametri inserire print(tokenizer.chat_template
    enable_thinking=False
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


In [4]:
# Guardiamo come è fatto il testo dopo aver applicato il chat template
text

'<|im_start|>system\n## Metadata\n\nKnowledge Cutoff Date: June 2025\nToday Date: 09 February 2026\nReasoning Mode: /no_think\n\n## Custom Instructions\n\nYou are a helpful AI assistant named SmolLM, trained by Hugging Face.\n\n<|im_start|>user\nGive me a brief explanation of gravity in simple terms in Italian language.<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n'

In [5]:
model_inputs

{'input_ids': tensor([[128011,   9125,    198,    567,  34689,    271,  81434,    356,  28540,
           2696,     25,   5651,    220,   2366,     20,    198,  15724,   2696,
             25,    220,   2545,   7552,    220,   2366,     21,    198,  26197,
            287,  14904,     25,    611,   2201,   5978,    771,    271,    567,
           8572,  39397,    271,   2675,    527,    264,  11190,  15592,  18328,
           7086,   4487,    337,  11237,     11,  16572,    555,    473,  36368,
          19109,    382, 128011,    882,    198,  36227,    757,    264,  10015,
          16540,    315,  24128,    304,   4382,   3878,    304,  15155,   4221,
             13, 128012,    198, 128011,  78191,    198, 128002,    271, 128003,
            198]], device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1

In [6]:
#Applichiamo il modello

results = model.generate(**{k: v.to(device) for k,v in model_inputs.items()}, max_new_tokens = 12000)

In [7]:
results

tensor([[128011,   9125,    198,    567,  34689,    271,  81434,    356,  28540,
           2696,     25,   5651,    220,   2366,     20,    198,  15724,   2696,
             25,    220,   2545,   7552,    220,   2366,     21,    198,  26197,
            287,  14904,     25,    611,   2201,   5978,    771,    271,    567,
           8572,  39397,    271,   2675,    527,    264,  11190,  15592,  18328,
           7086,   4487,    337,  11237,     11,  16572,    555,    473,  36368,
          19109,    382, 128011,    882,    198,  36227,    757,    264,  10015,
          16540,    315,  24128,    304,   4382,   3878,    304,  15155,   4221,
             13, 128012,    198, 128011,  78191,    198, 128002,    271, 128003,
            198,   8921,  29059,  24892,  11676,   5203,    369,   4458,   3091,
           1651,   9008,  29032,   7500,  97472,  92965,   3900,  41987,   1891,
            653,  66734,   3148,  48738,    822,     11,   2586,   1208,  50526,
             13,    763,  49

In [8]:
#De tokenizziamo
text_result = tokenizer.decode(results.squeeze(0).detach().cpu(), max_len=12000)
#Il modello restituisce gli id delle parole generate

In [12]:
print(text_result[len(text):])

La gravità è una forza che attira gli oggetti verso il centro di un corpo massiccio, come la Terra. In parole semplici, la Terra è così grande che attira tutti gli oggetti verso di sé, anche quelli molto piccoli, come le pietre o le persone. Questa forza ci fa cadere quando lasciamo andare qualcosa o quando tocchiamo terra.<|im_end|>
